# Inspector: del archivo al borrador de mapping (P0-B1)
Todo es local. El Inspector perfila el archivo, analiza **en tu máquina** los valores de la URL y propone un mapping que **tú revisas y apruebas**. Los ejemplos salen enmascarados.

In [ ]:
%load_ext autoreload
%autoreload 2
from dfir_copilot.profiling import inspect_source

RUTA = "/workspace/data/raw/three_months.csv"   # CSV, TSV, JSON/NDJSON o Parquet (también .gz)
IDIOMA = "es"

%time borrador = inspect_source(RUTA, lang=IDIOMA)
print(borrador.render())

## Revisa el borrador
Guárdalo, ábrelo, corrige lo que haga falta (zona horaria, nombres, roles) y **ese** archivo es el mapping aprobado. Mira sobre todo las decisiones y cualquier aviso de cobertura (`derived_coverage_gap`).

In [ ]:
RUTA_BORRADOR = "/workspace/data/mappings_borradores/three_months.yaml"
borrador.save(RUTA_BORRADOR)
print(open(RUTA_BORRADOR, encoding="utf-8").read())

## Ingerir en un caso nuevo con el mapping aprobado
Un caso = un dataset: usa un identificador nuevo.

In [ ]:
from dfir_copilot.cases import CaseWorkspace

CASO = "IDOR-INVOICES-2020Q4-INSPECTOR"
ws = CaseWorkspace.open_or_create(CASO, root="/workspace/data/cases", analyst="eder", lang=IDIOMA)
ws.add_raw(RUTA, link=True)
if not ws.meta["dataset"]:
    manifest = ws.ingest(RUTA, RUTA_BORRADOR)
    print(manifest["output"]["rows"], "filas |", manifest["roles"], "|", manifest["warnings"])
else:
    print("ya ingerido:", ws.meta["dataset"]["rows"], "filas")

## Integración: detectores sobre el caso nuevo
Abre el ledger (sella el dataset) y ejecuta los mismos detectores que usaste en la investigación original, ahora con el mapping generado por el Inspector.

In [ ]:
from dfir_copilot.tools.toolkit import Toolkit

engine = ws.engine()
ledger = ws.ledger(engine)
salida = Toolkit(engine, ledger).call("run_detectors")
print("ok:", salida.ok, "| roles:", engine.manifest["roles"])
print("detectores:", [(d["name"], d["status"], d["findings"]) for d in salida.data["detectors"]])
print("candidatos:", len(salida.data["candidates"]), [(c["signals"], c["severity"]) for c in salida.data["candidates"]])

## ¿Reproduce la investigación original?
Los identificadores de hallazgo son deterministas: si el mapping automático extrae lo mismo que el escrito a mano, salen los mismos. Solo **lee** el ledger original, no escribe en él.

In [ ]:
import json
from pathlib import Path

ORIGINAL_LEDGER = Path("/workspace/data/ledger/IDOR-INVOICES-2020Q4.jsonl")
if ORIGINAL_LEDGER.exists():
    lineas = (json.loads(l) for l in ORIGINAL_LEDGER.read_text(encoding="utf-8").splitlines() if l.strip())
    ids_original = {e["data"]["finding_id"] for e in lineas if e["type"] == "finding"}
    ids_nuevo = {e["data"]["finding_id"] for e in ledger.entries("finding")}
    print("hallazgos original:", len(ids_original), "| este caso:", len(ids_nuevo),
          "| comunes:", len(ids_original & ids_nuevo), "| solo aquí:", len(ids_nuevo - ids_original),
          "| solo en el original:", len(ids_original - ids_nuevo))
else:
    print("no hay ledger original en", ORIGINAL_LEDGER)

## Cadena de custodia

In [ ]:
print(ws.verify().render(IDIOMA))
print(ledger.summary()["counts"])